In [8]:
#required imports
from sklearn.metrics import roc_auc_score
import glob, os, math, sys,json,time
import numpy as np, pandas as pd, torch
from pathlib import Path

In [18]:
#declare gene-drug map
single_drugs = {
    "rifampicin" : ["rpoB"],
    "pyrazinamide": ["pncA"],
    "capreomycin" : ["tlyA"],
    "amikacin"    : ["eis"],
    "moxifloxacin": ["gyrA"],
    "levofloxacin": ["gyrB"],
}

multi_drugs = {
    "streptomycin": ["rpsL", "gid"],
    "isoniazid"   : ["katG", "inhA"],
    "ethionamide" : ["ethA", "ethR"],
    "ethambutol"  : ["embA", "embB", "embC"],
}

all_drugs = {**single_drugs, **multi_drugs}   # merge dicts

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Step 1: convert to float16 memory mapping and meta files for efficient memory

In [ ]:
import tqdm
# ──────────────────────────────────────────────────────────────
# 0.  one‑time conversion: NPZ  →  float16 mem‑map + meta
# ──────────────────────────────────────────────────────────────
#     *.npz files that hold token tensors  →  lightweight
#     float-16 memory-mapped arrays + a tiny “meta” file.
#
#     Why ?
#       • fp16 cuts RAM / disk roughly in half.
#       • mmap lets PyTorch stream slices straight off disk.
# ──────────────────────────────────────────────────────────────
def npz_to_memmap(gene):
    src_glob = f"data/embeddings/{gene}/token/{gene}_tok_chunk_*.npz"
    for npz in tqdm.tqdm(sorted(glob.glob(src_glob)), desc="convert"):
        meta_out = npz.replace(".npz", "_meta.npz")
        mmap_out = npz.replace(".npz", ".mmap")
        if os.path.exists(meta_out):        # skip if done
            continue

        z = np.load(npz)
        if "tokens" not in z:                # nothing to convert → skip
            print(f"  ⤷ skip {os.path.basename(npz)} (no 'tokens')")
            continue
        ids  = z["identifier"].astype(str)
        tok  = z["tokens"].astype("float16")      # cast to fp16 (½ RAM)
        mm   = np.memmap(mmap_out, mode="w+", dtype="float16",
                         shape=tok.shape)
        mm[:] = tok                             # write once
        mm.flush()
        np.savez_compressed(meta_out,
                            identifier=ids,
                            shape=tok.shape,
                            mmap_path=os.path.basename(mmap_out))
    print(f"{gene}: conversion finished")

# run once:
# npz_to_memmap("rpoB")

In [ ]:
import time
for drug, genes in all_drugs.items():
    print(f"\n===== {drug.upper()} =====")
    for g in genes:
        t0 = time.perf_counter()
        npz_to_memmap(g)
        dt = time.perf_counter() - t0
        print(f"  {g}: conversion finished in {dt/60:.2f} min")

## Step 2: PCA 

In [4]:
from sklearn.decomposition import IncrementalPCA
from tqdm import tqdm 

In [ ]:
# ──────────────────────────────────────────────────────────────
# joint PCA fit (only if .npz not yet present)
# 1)  Fit ONE PCA that sees all channels (320-D) from every gene
#     in the drug’s panel.  We save that basis once and re-use.
# ──────────────────────────────────────────────────────────────
def fit_joint_pca(genes, K, batch_res=10_000, root="data/embeddings", overwrite=False):
    """
    genes        : ['katG', 'inhA']  (order doesn’t matter)
    K            : how many PCs we want (e.g. 10)
    batch_res    : how many (L,320) rows to stream per partial_fit
    root         : top-level folder that holds the token chunks
    overwrite    : set True if we want to re-fit even when a file exists

    Output: writes   <root>/<katG_inhA>_pc10.npz   with
                 • mean      (shape 320)
                 • components (shape K×320)
    """

    #---- filename that will hold the PCA weights-----
    tag        = "_".join(genes) # 'katg_inhA
    comp_path  = Path(root) / f"{tag}_pc{K}.npz"   # katG_inhA_pc10.npz
    
    # --- skip if we already have it -----
    if comp_path.exists() and not overwrite:
        print("PCA already fitted →", comp_path)
        return comp_path

    # ---------------
    # ipca = IncrementalPCA(...)
    # ---------------
    # IncrementalPCA lets us feed data in mini-batches instead of forming one big (ΣL x 320) matrix in RAM. 
    # It keeps running estimates of 
    #  • the global mean (for centering)
    #  • the top-k eigenvectors (principal components)
    # every call to partial_fit(batch) updates those estimates.

    ipca = IncrementalPCA(n_components=K, batch_size=batch_res)

    # -----------------------------------------------------------------------
    # Walk through every *.mmap  file for every gene, stream rows, update PCA
    # -----------------------------------------------------------------------

    for g in genes:
        meta_paths = glob.glob(f"{root}/{g}/token/*_meta.npz")
        for mp in tqdm(meta_paths, desc=f"PCA fit {g}"):
            meta = np.load(mp, allow_pickle=True) # header info only
            mm   = np.memmap(mp.replace("_meta.npz", ".mmap"),
                             dtype="float16", mode="r", shape=tuple(meta["shape"])) # (N_isolates, L, 320)
            # loop over isolates inside this chunk
            for arr in mm:                         # arr: (L,320)
                # cast to float32 (IncrementalPCA needs FP32/64)
                ipca.partial_fit(arr.astype("float32"))
                #   ^ updates running mean & components using this batch
                #     NO full matrix ever lives in memory

    # Save the fitted basis → mean (320,)  and  components (K,320)
                
    np.savez(comp_path, 
             mean=ipca.mean_, # global channel-wise mean
             components=ipca.components_) # top-K eigenvectors
    print("joint PCA saved to", comp_path)
    return comp_path



In [ ]:
# ──────────────────────────────────────────────────────────────
# 2)  Take that PCA basis & project every chunk for ONE gene.
#     We stash the results under   .../token/PCA/
# ──────────────────────────────────────────────────────────────
def project_gene_to_pca(gene, comp_file, K, root="data/embeddings"):
    """
    gene       : e.g. 'gid'
    comp_file  : output of fit_joint_pca (…) – contains μ  &  W (K×320)
    K          : # components we kept
    root       : base folder that has   {gene}/token/*.mmap
    ----------------------------------------------------------------------------
    For every token chunk of <gene>:
        1) read the float16   (N_isolates , L , 320)   mmap
        2) subtract μ and right-multiply by Wᵀ  -->  (N , L , K)
        3) write a *new* float16 mmap into   {gene}/token/PCA/
           plus a tiny *_meta.npz   describing the file.
    ----------------------------------------------------------------------------
    Skips a chunk automatically if both output files already exist.
    """
    # ---------- 0) load PCA weights -----------------------------------------
    comp = np.load(comp_file)
    W, mu = comp["components"], comp["mean"]             # (K,320)  (320,)

    # ---------- 1) iterate over every *.mmap of this gene -------------------
    meta_paths = sorted(Path(f"{root}/{gene}/token").glob("*_meta.npz"))
    for mp in tqdm(meta_paths, desc=f"proj {gene}"):

        # ---------- target filenames inside ./PCA/ --------------------------
        pca_dir  = mp.parent / "PCA"             # …/gid/token/PCA/
        pca_dir.mkdir(exist_ok=True)

        base      = mp.stem.replace("_meta", "")              # gid_tok_chunk_0
        out_meta  = pca_dir / f"{base}_pc{K}_meta.npz"
        out_mmap  = pca_dir / f"{base}_pc{K}.mmap"

        # ---- skip if both files exist and sizes match ----------
        if out_meta.exists() and out_mmap.exists():
            continue
            
        # ---------- 2) open source mmap -------------------------------------
        meta  = np.load(mp, allow_pickle=True)
        mm_in = np.memmap(mp.with_suffix(".mmap"), dtype="float16", mode="r",
                          shape=tuple(meta["shape"])) # (N,L,320)
        
        # ---------- 3) allocate destination mmap ---------------------------
        mm_out = np.memmap(out_mmap, dtype="float16", mode="w+",
                           shape=(meta["shape"][0], # N isolates
                                  meta["shape"][1],  # L residues
                                  K)) # K PCs

        # ---------- 4) stream over isolates, project each ------------------
        for i in range(meta["shape"][0]): # loop rows
            X = mm_in[i].astype("float32") - mu # centre (L,320)
            mm_out[i] = (X @ W.T).astype("float16") # → (L,K) stored fp16

        mm_out.flush()  # ensure data is on disk
        
        # ---------- 5) tiny header so DataLoader knows the shape -----------
        np.savez(out_meta,
                 identifier=meta["identifier"], # same order as source
                 shape=mm_out.shape,
                 mmap_path=out_mmap.name) # just basename, not abs path




In [ ]:
# ------------------------------------------------------------
# 3) run for every drug (single & multi)
K = 10
for drug, genes in all_drugs.items():
    # 1) fit one PCA on the concatenated (320-D) channel space
    #    using ALL sequences from all genes in *this* drug group
    comp_file = fit_joint_pca(genes, K)  # works for single genes too  # returns path to .npz
    # 2) project every gene’s token mem-maps into that PCA basis
    for g in genes:
        project_gene_to_pca(g, comp_file, K)
    print(f" {drug}: PCA done\n")


## Step 3: multi gene data processing

In [6]:
from tqdm.auto import tqdm

# ──────────────────────────────────────────────────────────────
# helper ─ build a {isolate-id → label} dict for a *set* of genes
#          0 = susceptible (S) , 1 = resistant (R)
# ──────────────────────────────────────────────────────────────
def build_label_map(genes):
    dfs=[]
    for g in genes:
        csv=f"data/sequence_data_csv/{g}_combined_sequence_data.csv"
        d = pd.read_csv(csv, usecols=["Filename","Phenotype"])
        d = d.rename(columns={"Filename":"id", "Phenotype":f"ph_{g}"})
        dfs.append(d)
    # keep only isolates present in *all* genes
    lab = dfs[0]
    for d in dfs[1:]:
        lab = lab.merge(d, on="id", how="inner")

    #require the phenotype to *agree* across genes
    ph_cols=[c for c in lab if c.startswith("ph_")]
    lab   = lab[lab[ph_cols].nunique(axis=1)==1]   # unanimous

    # create dictionary → {'SAMN123': 0, 'SAMEA456':1, ...}
    return dict(zip(lab.id.astype(str),
                    (lab[ph_cols[0]]=="R").astype(int)))

In [ ]:
# ──────────────────────────────────────────────────────────────
# Dataset #1 : raw 320-dim token embeddings, concatenated length-wise
# ──────────────────────────────────────────────────────────────
class MultiGeneConcatDataset(torch.utils.data.Dataset):
    """
    Concatenates per-gene token tensors on demand:
        genes = ["katG","inhA"]  →  x shape (320 , L_katG+L_inhA) →  (320 , 740+269)
    """
    def __init__(self, genes, label_map):
        self.genes  = genes
        self.label  = label_map
        
        # build a list of mem-map blocks per gene: open mem-maps per gene
        self.blocks = {}              # gene → [(ids , mm), …]
        for g in genes:
            metas = sorted(Path(f"data/embeddings/{g}/token").glob("*_meta.npz"))
            gene_blocks  = []
            for mp in metas:
                meta = np.load(mp, allow_pickle=True)
                mm   = np.memmap(str(mp).replace("_meta.npz", ".mmap"),
                                 dtype="float16", mode="r",
                                 shape=tuple(meta["shape"]))
                gene_blocks.append((meta["identifier"].astype(str), mm))
            self.blocks[g] = gene_blocks
        # common isolate ids
        id_sets = [set(id_ for blk in self.blocks[g] for id_ in blk[0]) for g in genes]
        self.ids = sorted(set.intersection(*id_sets))  #global index

    def __len__(self): return len(self.ids)

    # small helper: fetch one isolate from one gene
    def _row(self, gene, ident):
        for ids, mm in self.blocks[gene]:
            w = np.where(ids == ident)[0]
            if w.size:                                  # (L , 320) ➜ (320 , L)
                return torch.from_numpy(mm[w[0]].astype("float32")).t()   # → (320 , L)
        raise KeyError(f"{ident} missing in {gene}") 

    def __getitem__(self, idx):
        ident = self.ids[idx]
        parts = [self._row(g, ident) for g in self.genes]      # list of (320 , Lg)
        x     = torch.cat(parts, dim=1)                  # length-concat concat on length axis → (320 , ∑L)
        y     = torch.tensor(self.label[ident], dtype=torch.float32)
        return x, y


In [ ]:
# ──────────────────────────────────────────────────────────────
# Dataset #2 : the same idea, but after PCA compression to `k` dims
# ──────────────────────────────────────────────────────────────
class PcaMultiGeneConcatDataset(torch.utils.data.Dataset):
    """
    Returns PCA-compressed token tensors.
        genes = ["rpsL","gid"], k = 10  →
            x  shape (k , L_rpsL+L_gid)
    """
    def __init__(self, genes, label_map, k):
        self.genes, self.k = genes, k
        self.label_map = label_map

        # open PCA-compressed mem-maps  (live under .../token/PCA/)
        self.blocks = {}           # gene → list[(ids, memmap)]
        for g in genes:
            metas = sorted(
                Path(f"data/embeddings/{g}/token/PCA")
                .glob(f"*_pc{k}_meta.npz")          # ← look only in PCA/
            )
            if not metas:
                raise FileNotFoundError(f"No *_pc{k}_meta.npz for {g}")
            g_blocks = []
            for mp in metas:
                meta = np.load(mp, allow_pickle=True)
                mmap_path = mp.parent / mp.name.replace("_meta.npz", ".mmap")
                mm = np.memmap(mmap_path, dtype="float16",
                               mode="r", shape=tuple(meta["shape"]))
                g_blocks.append((meta["identifier"].astype(str), mm))
            self.blocks[g] = g_blocks

        # identifiers present in *all* genes
        id_sets = [set(id_ for blk in self.blocks[g] for id_ in blk[0])
                   for g in genes]
        self.ids = sorted(set.intersection(*id_sets))

    def __len__(self): return len(self.ids)

    def _row(self, g, ident):
        for ids, mm in self.blocks[g]:
            w = np.where(ids == ident)[0]
            if w.size:
                return torch.from_numpy(mm[w[0]].astype("float32")).t()  # (k , Lg)
        raise KeyError(f"{ident} missing in {g}")

    def __getitem__(self, idx):
        ident = self.ids[idx]
        parts = [self._row(g, ident) for g in self.genes] # (k , Lg)
        x = torch.cat(parts, dim=1)                       # length-concat (k , ∑L)
        y = torch.tensor(self.label_map[ident], dtype=torch.float32)
        return x, y


## Step 4: CNN Model

In [ ]:
# ──────────────────────────────────────────────────────────────
# 2.  model inspired by MDCNN: I have used conv1D since we are not stacking multi gene embeddings. we are concatenating
# ──────────────────────────────────────────────────────────────
import torch.nn as nn, torch.nn.functional as F, math

# ──────────────────────────────────────────────────────────────
# ProteinCNN1x1 
# • Input :  (batch , C=in_dim , L_total)
#            C = 320   for raw tokens
#                =  K  for PCA-compressed channels
# • We CONCAT different genes *along the length axis*,
#   so we keep a **single 1-D convolutional stack**.
# ──────────────────────────────────────────────────────────────

class ProteinCNN1x1(nn.Module):
    def __init__(self, seq_len, in_dim=320, stem_out=64):
        """
        seq_len :  total padded length after gene-concat
        in_dim  :  #channels (320 or PCA-K)
        stem_out:  how many channels right after the 1×1 stem
        """
        super().__init__()
        # stem 1×1 conv – just re-mix the 320 (or K) channels
        self.stem = nn.Conv1d(in_dim, stem_out, 1)
        
        # ───── shallow CNN stack ───── 
        self.conv1 = nn.Conv1d(stem_out, 64, kernel_size=12, padding=6) # big receptive field
        self.pool1 = nn.MaxPool1d(3)                                    # shrink length ×3
        self.conv2 = nn.Conv1d(64, 32, 3, padding=1)
        self.conv3 = nn.Conv1d(32, 32, 3, padding=1)
        self.pool2 = nn.MaxPool1d(3)                                    # shrink again

        # flatten dim - figure out flatten size (once)
        with torch.no_grad():
            dummy = torch.zeros(1, in_dim, seq_len) # fake batch=1
            flat = self._forward_feat(dummy).numel() # total features

        # ───── dense head ─────
        self.fc1 = nn.Linear(flat, 256)
        self.fc2 = nn.Linear(256, 256)
        self.fc_out = nn.Linear(256, 1)       # final logit

    # ----------------------------------------------------------
    # internal helper: CNN part only
    # ----------------------------------------------------------
    
    def _forward_feat(self, x):
        x = self.stem(x)
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool2(x)
        return x
    # ----------------------------------------------------------
    # full forward pass
    # ----------------------------------------------------------
    def forward(self, x):
        """
        x shape  :  (batch , C=in_dim , L_total)
        returns  :  raw logits  (batch,)   – use torch.sigmoid later
        """
        x = self._forward_feat(x)  # CNN feature map
        x = torch.flatten(x, 1)   # keep batch dim, flatten rest
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc_out(x).squeeze(-1)   # (batch,)   logits


## step 5: Per-token and PCA runs

In [12]:
# ──────────────────────────────────────────────────────────────
#  Train token model (full 320‑dim or PCA‑K)  +  logging
# ──────────────────────────────────────────────────────────────
# High-level flow
#   1) Build the right Dataset  (single-gene ⟹ TokenMemmapMap
#                                multi-gene ⟹ MultiGeneConcat / PCA-variant)
#   2) Split → DataLoaders
#   3) Instantiate ProteinCNN1x1
#   4) Train w/ pos_weight + bias-freeze trick
#   5) Log AUC & ACC each epoch, save model + CSV history
# --------------------------------------------------------------
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, random_split


In [13]:
# -------------------------
# helper: pad-collate
# -------------------------

def pad_collate(batch, L_PAD):
    """Right-pad every sequence in the mini-batch to the same length."""
    xs, ys = zip(*batch)
    xs_pad = [F.pad(x, (0, L_PAD - x.shape[1])) if x.shape[1] < L_PAD else x
              for x in xs]
    return torch.stack(xs_pad), torch.stack(ys)

# ─────────────────────────────────────────────────────────
#  Map-style datasets that stream from float-16 mem-maps
# ─────────────────────────────────────────────────────────
#
# Why two classes?
#   • TokenMemmapMap → raw 320-channel ESM tokens
#   • PcaMemmapMap   → PCA-compressed version (k channels)
#
# Both expose   __len__   and   __getitem__   so they plug straight
# into a PyTorch DataLoader.
# --------------------------------------------------------
class TokenMemmapMap(torch.utils.data.Dataset):
    """
    Map‑style dataset: supports __len__, __getitem__
    Each item → (tensor  [320,L] float32,  label float32)
        Returns
        x : torch.FloatTensor  (320 , L)     # already transposed
        y : scalar torch.Tensor  (label 0/1)
    """
    def __init__(self, meta_paths, label_dict):
        # ---- unpack every “chunk” meta and open its mem-map ----------
        self.blocks = []          # list of (ids, memmap) per chunk
        self.lookup = []          # flat list of (block_idx, row_idx)

        
        for bidx, meta_path in enumerate(meta_paths):
            meta = np.load(meta_path, allow_pickle=True)
            ids  = meta["identifier"]             # list/array of isolate IDs
            shape = tuple(meta["shape"])          # (n_rows, L, 320)
            mmap_path = meta_path.replace("_meta.npz", ".mmap")
            mm = np.memmap(mmap_path, dtype="float16", mode="r", shape=shape) # memory-map (readonly)

            
            self.blocks.append((ids, mm))
            # extend lookup
            self.lookup.extend([(bidx, r) for r in range(shape[0])])
        self.label_dict = label_dict

    def __len__(self):
        return len(self.lookup)

    def __getitem__(self, idx):
        # translate global idx → which chunk / which row
        bidx, ridx = self.lookup[idx]
        ids, mm = self.blocks[bidx]
        seq_id   = ids[ridx]
        # slice mem-map row, cast to fp32, transpose to (channels , length)
        x = torch.from_numpy(mm[ridx].astype("float32")).t()  # (320,L)
        y = torch.tensor(self.label_dict[seq_id], dtype=torch.float32)
        return x, y



class PcaMemmapMap(torch.utils.data.Dataset):          # PCA‑K
    """
    Same idea but for PCA-compressed matrices.
        • in_dim = k (e.g. 10)
        • underlying shape each row: (L , k)  → we transpose to (k , L)
    """
    def __init__(self, meta_paths, label_dict, k):
        self.blocks = [] # [(ids, memmap_k), …]
        self.lookup = []
        self.k = k
        for bidx, p in enumerate(meta_paths):
            m  = np.load(p, allow_pickle=True)
            # open the companion .mmap (same basename, different suffix)
            mm = np.memmap(p.replace("_meta.npz", ".mmap"),
                           dtype="float16", mode="r", shape=tuple(m["shape"])) # (n , L , k)
            self.blocks.append((m["identifier"], mm))
            self.lookup += [(bidx, r) for r in range(m["shape"][0])]
        self.label = label_dict
        
    def __len__(self):  return len(self.lookup)
        
    def __getitem__(self, idx):
        b,r = self.lookup[idx]; ids,mm = self.blocks[b]
        x = torch.from_numpy(mm[r].astype("float32")).t()   # (K,L)
        y = torch.tensor(self.label[ids[r]], dtype=torch.float32)
        return x, y

In [14]:
# ----- main training function -----------------------------------------------
def train_token(
    gene,                    # “rpoB”  or  “isoniazid” (drug key)
    in_dim=320,              # 320 → full tokens ; k → PCA-k channels
    batch_size=32,
    n_epochs=20,
    lr=5e-4,
    freeze_bias_frac=0.25,    # how long to freeze output-bias
    val_frac=0.2,
    out_root="runs"):


    
    t0 = time.time()
    run_dir = f"{out_root}/{gene}_dim{in_dim}"
    os.makedirs(run_dir, exist_ok=True)

    # ------------------------------------------------------------
    # 1) decide which Dataset class to use
    # ------------------------------------------------------------
    
    if gene in multi_drugs:               # ── multi-gene branch ─────────────────
        genes = multi_drugs[gene]         # e.g. ["katG","inhA"]
        label_map = build_label_map(genes)     # unanimous R/S
        # full_ds   = MultiGeneConcatDataset(genes, label_map)
        # in_dim    = 320                                   # channels unchanged

        if in_dim == 320:        # raw 320-channel tokens
            full_ds = MultiGeneConcatDataset(genes, label_map)
        else:                    # PCA-K channels
            full_ds = PcaMultiGeneConcatDataset(genes, label_map, k=in_dim)

    
    else:                              # ── single-gene branch (old behaviour) ──
        ph = pd.read_csv(f"data/sequence_data_csv/{gene}_combined_sequence_data.csv",usecols=["Filename","Phenotype"])
        label_map = dict(zip(ph.Filename.astype(str),(ph.Phenotype=="R").astype(int)))
    
        if in_dim == 320:                              # raw tokens
            metas = [p for p in glob.glob(
                         f"data/embeddings/{gene}/token/*_meta.npz")
                     if "_pc" not in p]
            full_ds = TokenMemmapMap(metas, label_map)
        else:                                          # PCA-K mem-maps
            metas = glob.glob(
                f"data/embeddings/{gene}/token/PCA/*_pc{in_dim}_meta.npz")
            full_ds = PcaMemmapMap(metas, label_map, k=in_dim)
        assert metas, "No meta files found – check path or in_dim"


    # ------------------------------------------------------------
    # 2) make train / val split + loaders
    # ------------------------------------------------------------
    labels_arr = np.fromiter(label_map.values(), dtype=np.int32)
    nR = labels_arr.sum()              # resistant = 1
    nS = len(labels_arr) - nR          # susceptible = 0

    val_len  = int(len(full_ds)*val_frac)
    train_len= len(full_ds)-val_len

    # quick probe to find the longest length we’ll see (for padding)
    probe_n = min(100, len(full_ds))  # <- cap at dataset size
    # L_PAD = max(full_ds[i][0].shape[1] for i in range(100))  # cheap probe
    L_PAD = max(full_ds[i][0].shape[1] for i in range(probe_n)) 

    tr_ds, va_ds = random_split(full_ds, [train_len, val_len],generator=torch.Generator().manual_seed(0))
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,num_workers=2, pin_memory=True,collate_fn=lambda b: pad_collate(b, L_PAD))
    va_ld = DataLoader(va_ds, batch_size=batch_size, shuffle=False,num_workers=2, pin_memory=True,collate_fn=lambda b: pad_collate(b, L_PAD))

    # model
    # L_PAD = full_ds[0][0].shape[-1]
    stem_out = 64 if in_dim==320 else 32
    model = ProteinCNN1x1(seq_len=L_PAD, in_dim=in_dim,stem_out=stem_out).to(device)

    # class-imbalance weight
    # pos_weight = torch.tensor((ph.Phenotype=='S').sum() /(ph.Phenotype=='R').sum()).to(device)
    pos_weight = torch.tensor(nS / nR, dtype=torch.float32).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    #
    # init output bias to log-odds, then freeze for first X% epochs
    freeze_ep = max(1,int(n_epochs*freeze_bias_frac))
    with torch.no_grad():
        # pR=(ph.Phenotype=='R').mean()
        pR = nR / (nR + nS + 1e-8)              # prevalence of “R”
        # model.fc_out.bias.fill_(math.log(pR/(1-pR+1e-8)))
        model.fc_out.bias.fill_(math.log(pR / (1 - pR)))
    model.fc_out.bias.requires_grad=False



    # ------------------------------------------------------------
    # 4) training loop
    # ------------------------------------------------------------

    hist=[]
    for ep in range(1,n_epochs+1):
        # train
        model.train()
        for xb,yb in tr_ld:
            xb,yb=xb.to(device), yb.to(device)
            opt.zero_grad(); loss_fn(model(xb),yb).backward(); opt.step()
            
        # un-freeze bias after the warm-up
        if ep==freeze_ep: model.fc_out.bias.requires_grad=True

        # val
        model.eval(); prob,yt=[],[]
        with torch.no_grad():
            for xb,yb in va_ld:
                prob.append(torch.sigmoid(model(xb.to(device))).cpu())
                yt.append(yb)
        prob=np.concatenate([p.numpy() for p in prob])
        yt  =np.concatenate([y.numpy() for y in yt])
        auc = roc_auc_score(yt, prob)
        acc = ((prob>0.5)==yt).mean()
        print(f"ep {ep:02d}/{n_epochs}  val_auc={auc:.3f}  val_acc={acc:.3f}")
        hist.append({"epoch":ep,"val_auc":float(auc),"val_acc":float(acc)})

    # save
    torch.save(model.state_dict(), f"{run_dir}/{gene}_model.pt")
    pd.DataFrame(hist).to_csv(f"{run_dir}/{gene}_history.csv", index=False)

    print(f" done — model & history saved to {run_dir}  "
          f"({(time.time()-t0)/60:.1f} min)")
    return model, tr_ds, va_ds, pd.DataFrame(hist)

In [ ]:
# model_pca10, tr_ds, va_ds, hist=train_token("gyrB", in_dim=10)  

# ------------------------------------------------------------------
# 0. Which experiments do you want to run?
#    Each row:  (drug_or_gene , in_dim , note)
# ------------------------------------------------------------------
TODO = [
    # ---------- single-gene runs ----------
    ("gyrB",        10,  "PCA-10"),
    ("gyrB",       320,  "raw-320"),
    ("eis",        320,  "raw-320"),
    # ---------- multi-gene drugs ----------
    ("isoniazid",  320,  "katG+inhA raw-320"),
    ("streptomycin", 10, "rpsL+gid  PCA-10"),
]

# ------------------------------------------------------------------
# 1. Loop and train: just editing to do is enough
# ------------------------------------------------------------------
results = {}                     # keep a handle on every model/hist if needed

for target, in_dim, tag in TODO:
    print(f"\n=== {target}  |  {tag}  ===============================")
    model, tr_ds, va_ds, hist = train_token(
        gene       = target,
        in_dim     = in_dim,
        batch_size = 32,
        n_epochs   = 20,
        lr         = 5e-4,
    )
    results[(target, in_dim)] = {"model": model, "hist": hist}


## Step 6: SHAP residue

In [ ]:
import random, shap
import torch.nn as nn


#  Tiny wrapper : SHAP expects the model to output (B, #classes)
#  Our CNN returns a 1-D logit tensor (B,) → unsqueeze to (B,1)


class Wrapped(nn.Module):
    def __init__(self, base):          # base =  trained model
        super().__init__()
        self.base = base
    def forward(self, x):
        return self.base(x).unsqueeze(1)    # (B,) → (B,1) for SHAP

        

# ------------------------------------------------------------------
#  shap_per_residue
#  -----------------
#  Works for BOTH single-gene (per_gene_lengths = None)
#  and multi-gene (pass lengths + names).
#  Returns a tidy DataFrame:
#     sample_idx , label , importance_full ,
#     importance_<gene1> , importance_<gene2> , …

def shap_per_residue(model,
                     train_ds, val_ds,      #datasets already split
                     background_size, explain_samples,
                     per_gene_lengths=None,   # # e.g. [740,269]  or None
                     gene_names=None,         # ["katG","inhA"] matching ^
                     device="cuda"):
    
    # ----------------------------------------------------------
    # 0) put model on the right device, inference-only mode
    # ---------------------------------------------------------
    model = model.to(device).eval()

    # ----------------------------------------------------------
    # 1) pick SHAP background  (random B samples from training)
    # ----------------------------------------------------------
    B = max(1, min(background_size, len(train_ds)))             # clamp to dataset size
    bg_idx = random.sample(range(len(train_ds)), B)
    background = torch.stack([train_ds[i][0] for i in bg_idx]).to(device)
    
    
    # DeepExplainer needs model wrapped to produce 2-D outputs
    explainer = shap.DeepExplainer(Wrapped(model), [background])

    # ----------------------------------------------------------
    # 2) choose E samples from the validation set to explain
    # ----------------------------------------------------------
    E = max(1, min(explain_samples, len(val_ds)))
    samp_idx = random.sample(range(len(val_ds)), E)
    xs       = torch.stack([val_ds[i][0] for i in samp_idx]).to(device)
    ys       = [val_ds[i][1] for i in samp_idx]

    # ----------------------------------------------------------
    # 3) compute SHAP values
    #    sv: (E , C=channels , L_total)
    #    importance = Σ_c |SHAP|  → (E , L_total)
    # ----------------------------------------------------------
    sv  = explainer.shap_values([xs], check_additivity=False)[0]  # (E,C,L)
    imp = np.abs(sv).sum(axis=1)                                  # (E,Ltot)


    # Core columns every DataFrame has
    out = {
        "sample_idx"      : samp_idx,
        "label"           : [int(y) for y in ys],
        "importance_full" : list(imp)  # each row: ndarray (L_total,)
    }

    
    # ----------------------------------------------------------
    # 4) If multi-gene, slice the long vector into per-gene chunks
    # ----------------------------------------------------------
    if per_gene_lengths is not None:
        assert gene_names is not None, "need gene_names with per_gene_lengths"
        cuts = np.cumsum([0] + per_gene_lengths)                  # [0, 740, …]
        for gi, g in enumerate(gene_names):
            # list of ndarrays, one per isolate, each (L_gene,)
            out[f"importance_{g}"] = [imp[n, cuts[gi]:cuts[gi+1]]
                                       for n in range(E)]

    return pd.DataFrame(out)


In [ ]:
# ------------------------------------------------------------
# 1. configuration ── select the drug to explain
# ------------------------------------------------------------

DRUG      = "levofloxacin"     # << change only this line
GENES     = all_drugs[DRUG]  # e.g. ["rpsL","gid"]

# ------------------------------------------------------------
# 2. recover the very dataset already used for training
#    (tr_ds  &  va_ds  come from  earlier `train_token`)

# here we assume they’re already in memory:
assert isinstance(tr_ds, torch.utils.data.Dataset)
assert isinstance(va_ds, torch.utils.data.Dataset)

# figure-out per–gene lengths once, from the first isolate ---------
per_gene_len = []
for g in GENES:
    # first mem-map of that gene → meta → shape[1] is length
    mp = next(Path(f"data/embeddings/{g}/token").glob("*_meta.npz"))
    Lg = int(np.load(mp, allow_pickle=True)["shape"][1])
    per_gene_len.append(Lg)        # e.g. [740, 269] for katG+inhA

print("lengths:", dict(zip(GENES, per_gene_len)))   # sanity check

bg_size = int(0.8 * len(tr_ds))
exp_size = int(0.8 * len(va_ds))

# single-gene run
# df_single = shap_per_residue(model_pca10, tr_ds, va_ds,
#                              background_size=100, explain_samples=200)

# multi-gene run (katG+inhA)


shap_df = shap_per_residue(
    model_pca10,
    train_ds=tr_ds,
    val_ds=va_ds,
    background_size=100,
    explain_samples=200,
    per_gene_lengths=per_gene_len,
    gene_names=GENES,          # order must match per_gene_len
    device="cuda"              # or "cpu"
)

out_file = f"metrics_output/{DRUG}_shap_per_residue.pkl"
shap_df.to_pickle(out_file, protocol=4)
print(" SHAP saved to", out_file)

# ------------------------------------------------------------
# 5. quick demo: top-10 residues per gene in the *first* explained sample
# ------------------------------------------------------------
row = shap_df.iloc[0]

for g, L in zip(GENES, per_gene_len):
    imp = row[f"importance_{g}"].squeeze()      # (L,) vector
    top = np.argsort(imp)[-10:][::-1]           # indices with largest |SHAP|
    print(f"{g} 0-based:", top)
    print(f"{g} 1-based:", top + 1, "\n")       # if you prefer 1-based

## Step 7: Precision and Recall

In [15]:
# ──────────────────────────────────────────────────────────────
# 0. CONFIG — EDIT ONLY THE DRUG NAME (everything else auto-fills)
# ──────────────────────────────────────────────────────────────
DRUG  = "levofloxacin"                         # <-- choose drug
K_VALUES        = [1, 5, 10]                   # precision / recall cut-offs
ALLOWED_CONF    = ['1) Assoc w R', '2) Assoc w R - Interim']

DATA_DIR        = Path("data")                 # repo data root
SHAP_PKL        = Path(f"metrics_output/{DRUG}_shap_per_residue.pkl")
CATALOG_CSV     = DATA_DIR / "filtered_variants_output.csv"
OUT_DIR         = Path("metrics_output")
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [16]:
# ------------------------------------------------------------------
# 1.  Load SHAP dataframe & WHO catalogue
# ------------------------------------------------------------------
print("• loading SHAP & catalogue …")
shap_df  = pd.read_pickle(SHAP_PKL)            # SHAP per-residue scores
catalog  = pd.read_csv(CATALOG_CSV)            # WHO catalogue
catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1   # 0-based once

• loading SHAP & catalogue …


In [19]:
GENES = all_drugs[DRUG]

In [ ]:
# ------------------------------------------------------------------
# 3.  Utility helpers (short & self-documented)
# ------------------------------------------------------------------
def mean_abs_importance(df, col):
    """
    Compute mean(|SHAP|) for every residue in <col>.
    Works if arrays are saved as (L,), (L,1) or (1,L).
    Returns a rank-ordered DataFrame.
    """
    arr_list = [np.asarray(a).squeeze() for a in df[col].to_list()]  # (N,L)
    stack    = np.stack(arr_list, axis=0)
    mean_imp = stack.mean(axis=0)
    return (pd.DataFrame({"Residue_Position": np.arange(len(mean_imp)),
                          "MeanAbsSHAP": mean_imp})
            .sort_values("MeanAbsSHAP", ascending=False)
            .reset_index(drop=True))

def max_abs_importance(shap_df, col_name):
    """
    Rank residues by max |SHAP| across isolates.

    Parameters
    ----------
    shap_df   : DataFrame returned by `shap_per_residue`
    col_name  : e.g. 'importance_katG'

    Returns
    -------
    DataFrame sorted descending
        Residue_Position   0-based index
        MaxAbsSHAP         max(|SHAP|) across all isolates
    """
    # 1) list → ndarray list, each (L,) after squeeze
    arr_list = [np.asarray(v).squeeze() for v in shap_df[col_name].to_list()]

    # 2) stack to (N isolates × L)
    stack = np.stack(arr_list, axis=0)

    # 3) take max along the isolate axis
    max_imp = np.abs(stack).max(axis=0)          # (L,)

    # 4) tidy DataFrame
    return (pd.DataFrame({"Residue_Position": np.arange(len(max_imp)),
                          "MaxAbsSHAP": max_imp})
              .sort_values("MaxAbsSHAP", ascending=False)
              .reset_index(drop=True))


def rows_for_gene(cat, g):
    """Catalogue rows that correspond to <g> (case-insensitive)."""
    return cat[cat["gene"].str.lower() == g.lower()].copy()

def exclusion_set(cat_rows):
    """
    Positions *excluded* from denominator:
      • confidence == 3   OR
      • intersectional != True
    """
    mask_unc   = cat_rows["confidence"] == "3"
    mask_not_i = cat_rows["intersectional"] != True
    return set(cat_rows.loc[mask_unc | mask_not_i, "aa_pos_0idx"])

def bona_fide(cat_rows):
    """Positions confidently associated with resistance (conf 1/2 & intersectional)."""
    mask = (cat_rows["confidence"].isin(ALLOWED_CONF) &
            (cat_rows["intersectional"] == True))
    return set(cat_rows.loc[mask, "aa_pos_0idx"])

def greedy_topk(rank_df, exclude, k):
    """
    Walk importance ranking top-down; pick first <k> sites *not* in <exclude>.
    """
    chosen = []
    for pos in rank_df["Residue_Position"]:
        if pos in exclude:
            continue
        chosen.append(pos)
        if len(chosen) == k:
            break
    return chosen
    
def gene_length(g):
    """Read length L from first *_meta.npz for gene <g>."""
    meta = next(Path(f"data/embeddings/{g}/token").glob("*_meta.npz"))
    return int(np.load(meta, allow_pickle=True)["shape"][1])

In [ ]:
# ------------------------------------------------------------------
# 4.  Auto-detect token length for each gene (nice to print)
# ------------------------------------------------------------------
PER_GENE_LENGTHS = [gene_length(g) for g in GENES]
print("gene lengths:", dict(zip(GENES, PER_GENE_LENGTHS)))

# ------------------------------------------------------------------
# 5.  Loop over genes → precision / recall table
# ------------------------------------------------------------------
all_rows = []

for gene in GENES:
    print(f"\n── evaluating {gene} ───────────────────────────")

    # 5-a) rank residues by max |SHAP|
    rank_df = max_abs_importance(shap_df, f"importance_{gene}")
    rank_df.to_csv(OUT_DIR / f"{DRUG}_{gene}_ranked_SHAP.csv", index=False)

    # 5-b) WHO catalogue facts
    cat_rows = rows_for_gene(catalog, gene)
    bona     = bona_fide(cat_rows)       # ground-truth resistance positions
    excl     = exclusion_set(cat_rows)   # sites to ignore in denominator
    n_true   = len(bona)

    # 5-c) evaluate at each K
    for k in K_VALUES:
        topk   = greedy_topk(rank_df, excl, k)
        k_eff  = len(topk)                             # < k if list exhausted
        tp     = len(bona.intersection(topk))          # true positives
        prec   = tp / k_eff if k_eff else 0.0
        rec    = tp / n_true if n_true else 0.0
        f1     = (2*prec*rec)/(prec+rec+1e-8) if (prec+rec) else 0.0

        # human-readable list of variants “hit” in top-k
        hits = (cat_rows[cat_rows["aa_pos_0idx"].isin(topk) &
                         cat_rows["confidence"].isin(ALLOWED_CONF) &
                         (cat_rows["intersectional"] == True)]
                .drop_duplicates("aa_pos_0idx")["variant"]
                .tolist() or ["None"])

        all_rows.append({"drug": DRUG,      "gene": gene,
                         "k_req": k,        "k_eff": k_eff,
                         "total_res_pos": n_true,
                         "TP": tp,          "precision": prec,
                         "recall": rec,     "F1": f1,
                         "hit_variants": ", ".join(hits)})

        print(f"  k={k:<2d}  picked={k_eff:<2d}  "
              f"TP={tp}/{n_true:<2d}  prec={prec:.2f}  "
              f"rec={rec:.2f}  F1={f1:.2f}")

# ------------------------------------------------------------------
# 6.  Save summary table
# ------------------------------------------------------------------
metrics_df = pd.DataFrame(all_rows)
out_csv    = OUT_DIR / f"{DRUG}_precision_recall.csv"
metrics_df.to_csv(out_csv, index=False)
print("\n finished →", out_csv)